# Customer Segmentation Analysis

This notebook performs end-to-end customer segmentation using K-Means clustering.

**Workflow:**
1. Upload your data file (CSV or Excel)
2. Automatic data evaluation and profiling
3. Data cleaning and preprocessing
4. K-Means clustering with optimal K selection
5. Cluster visualizations and interpretation

---

## 1. Setup & Imports

In [ ]:
import io
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')

# Global state shared across cells
state = {
    'raw_df': None,
    'clean_df': None,
    'feature_matrix': None,
    'cluster_labels': None,
    'feature_columns': None,
    'scaler': None,
    'optimal_k': None,
}

# Plot style
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
PALETTE = 'tab10'


# ---------------------------------------------------------------------------
# Data evaluation helpers — defined here so they are available as soon as
# the upload widget (Section 2) is rendered, regardless of cell run order.
# ---------------------------------------------------------------------------

def classify_column(series):
    """Determine column role: id, target, numeric, categorical, datetime, or drop."""
    name_lower = series.name.lower()
    nunique = series.nunique(dropna=True)
    n = len(series)

    # Likely an ID column
    if any(tok in name_lower for tok in ['id', 'uuid', 'key', 'index']) or nunique == n:
        return 'id'

    # Datetime
    if pd.api.types.is_datetime64_any_dtype(series):
        return 'datetime'
    if series.dtype == object:
        try:
            pd.to_datetime(series.dropna().head(20), infer_datetime_format=True)
            return 'datetime'
        except Exception:
            pass

    # Numeric
    if pd.api.types.is_numeric_dtype(series):
        if nunique <= 2:
            return 'binary'
        return 'numeric'

    # Categorical (low cardinality object)
    if series.dtype == object:
        if nunique / n < 0.5:
            return 'categorical'
        return 'high_cardinality_text'

    return 'unknown'


def evaluate_data(df):
    """Print a structured profile of the dataframe."""
    print('=' * 65)
    print('  DATA EVALUATION REPORT')
    print('=' * 65)
    print(f'  Shape : {df.shape[0]:,} rows  x  {df.shape[1]} columns')
    print(f'  Memory: {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
    print(f'  Duplicate rows: {df.duplicated().sum():,}')
    print()

    col_info = []
    for col in df.columns:
        s = df[col]
        missing = s.isna().sum()
        missing_pct = missing / len(s) * 100
        ctype = classify_column(s)
        col_info.append({
            'Column': col,
            'Dtype': str(s.dtype),
            'Role': ctype,
            'Unique': s.nunique(dropna=True),
            'Missing': missing,
            'Missing %': f'{missing_pct:.1f}%',
            'Sample Values': str(list(s.dropna().unique()[:3])),
        })

    report_df = pd.DataFrame(col_info)
    with pd.option_context('display.max_colwidth', 40, 'display.max_rows', 100):
        display(report_df)

    # Flag issues
    high_missing = report_df[report_df['Missing'] / len(df) > 0.5]
    if not high_missing.empty:
        print()
        print('WARNING: Columns with >50% missing values (will be dropped during cleaning):')
        print(' ', list(high_missing['Column']))

    numeric_cols = report_df[report_df['Role'] == 'numeric']['Column'].tolist()
    if numeric_cols:
        print()
        print('Numeric column statistics:')
        display(df[numeric_cols].describe().round(2))

    # Store column classification for use in cleaning
    state['col_roles'] = {row['Column']: row['Role'] for row in col_info}
    print()
    print('Evaluation complete. Run the Data Cleaning cell next.')


print('Libraries loaded successfully.')

---
## 2. Upload Data File

Upload a **CSV** or **Excel** (.xlsx / .xls) file containing your customer data.

In [ ]:
upload_widget = widgets.FileUpload(
    accept='.csv,.xlsx,.xls',
    multiple=False,
    description='Upload file',
    button_style='primary',
    layout=widgets.Layout(width='250px'),
)

status_label = widgets.Label('')
load_btn = widgets.Button(
    description='Load & Evaluate',
    button_style='success',
    icon='check',
    layout=widgets.Layout(width='180px'),
)
eval_output = widgets.Output()


def load_file(b):
    eval_output.clear_output()
    with eval_output:
        try:
            if not upload_widget.value:
                status_label.value = 'No file selected. Please upload a file first.'
                return

            # ipywidgets v8: .value is a tuple of dicts with keys 'name', 'content', etc.
            # ipywidgets v7: .value is a dict keyed by filename with nested 'metadata'/'content'.
            uploaded = upload_widget.value
            if isinstance(uploaded, dict):
                # v7 fallback
                file_info = list(uploaded.values())[0]
                filename = file_info['metadata']['name']
                content = bytes(file_info['content'])
            else:
                # v8 (tuple)
                file_info = uploaded[0]
                filename = file_info['name']
                content = bytes(file_info['content'])

            if filename.endswith('.csv'):
                df = pd.read_csv(io.BytesIO(content))
            elif filename.endswith(('.xlsx', '.xls')):
                df = pd.read_excel(io.BytesIO(content))
            else:
                status_label.value = 'Unsupported format. Use CSV or Excel.'
                return

            state['raw_df'] = df
            status_label.value = (
                f'Loaded "{filename}" — {df.shape[0]:,} rows x {df.shape[1]} columns'
            )
            evaluate_data(df)

        except Exception as e:
            import traceback
            status_label.value = f'Error: {e}'
            traceback.print_exc()


load_btn.on_click(load_file)

display(
    widgets.HBox([upload_widget, load_btn]),
    status_label,
    eval_output,
)

---
## 3. Data Evaluation

The cell below defines the evaluation logic called automatically after upload.  
You can also call `evaluate_data(state['raw_df'])` manually at any time.

In [ ]:
# classify_column() and evaluate_data() are defined in the Setup cell (Section 1)
# so they are always available when the upload button fires, regardless of run order.

# Manual re-evaluation — useful if you want to re-inspect after the upload widget.
if state['raw_df'] is not None:
    evaluate_data(state['raw_df'])
else:
    print('No data loaded yet. Upload a file in Section 2 above.')

---
## 4. Data Cleaning & Preprocessing

The following steps are applied automatically:
- Drop ID-like, datetime, and high-cardinality text columns
- Drop columns missing >50% of values
- Remove duplicate rows
- Impute missing numeric values with the **median**
- Impute missing categorical values with the **mode**
- One-hot-encode categorical columns
- Standard-scale all numeric features

In [ ]:
def clean_and_preprocess(df, col_roles):
    print('=' * 65)
    print('  DATA CLEANING & PREPROCESSING')
    print('=' * 65)
    n_original = len(df)

    # --- Step 1: Drop unhelpful columns ---
    drop_roles = {'id', 'datetime', 'high_cardinality_text', 'unknown'}
    drop_cols = [c for c, r in col_roles.items() if r in drop_roles]
    # Also drop columns missing >50%
    high_missing = [c for c in df.columns if df[c].isna().mean() > 0.5]
    drop_cols = list(set(drop_cols + high_missing))

    if drop_cols:
        print(f'Dropping columns: {drop_cols}')
        df = df.drop(columns=drop_cols)

    # --- Step 2: Remove duplicates ---
    n_before = len(df)
    df = df.drop_duplicates()
    removed = n_before - len(df)
    if removed:
        print(f'Removed {removed:,} duplicate rows')

    # --- Step 3: Identify remaining column types ---
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    cat_cols = df.select_dtypes(exclude='number').columns.tolist()

    # --- Step 4: Impute ---
    if numeric_cols:
        num_imputer = SimpleImputer(strategy='median')
        df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])
        print(f'Imputed {len(numeric_cols)} numeric column(s) with median')

    if cat_cols:
        cat_imputer = SimpleImputer(strategy='most_frequent')
        df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])
        print(f'Imputed {len(cat_cols)} categorical column(s) with mode')

    # --- Step 5: One-hot encode categoricals ---
    if cat_cols:
        df = pd.get_dummies(df, columns=cat_cols, drop_first=False)
        print(f'One-hot encoded {len(cat_cols)} categorical column(s)')

    # --- Step 6: Scale numeric features ---
    feature_cols = df.columns.tolist()
    scaler = StandardScaler()
    X = scaler.fit_transform(df[feature_cols])
    print(f'Scaled {len(feature_cols)} features with StandardScaler')

    print()
    print(f'Clean dataset: {len(df):,} rows x {len(feature_cols)} features')
    print(f'(Started with {n_original:,} rows)')
    print()
    print('Features used for clustering:')
    print(feature_cols)

    return df, X, feature_cols, scaler


if state['raw_df'] is None:
    print('No data loaded. Please upload a file first.')
else:
    clean_df, X, feature_cols, scaler = clean_and_preprocess(
        state['raw_df'].copy(),
        state.get('col_roles', {}),
    )
    state['clean_df'] = clean_df
    state['feature_matrix'] = X
    state['feature_columns'] = feature_cols
    state['scaler'] = scaler
    print()
    print('Preprocessing complete. Run the Clustering cell next.')

---
## 5. Find Optimal K (Elbow + Silhouette)

Two complementary methods are used to choose the best number of clusters:
- **Elbow method** — plots inertia (within-cluster sum of squares) vs K; look for the "elbow"
- **Silhouette score** — measures how well each point fits its cluster (higher = better, max 1.0)

In [ ]:
# Configure the K search range
K_MIN = 2
K_MAX = 10   # increase if you expect many segments


def find_optimal_k(X, k_min=2, k_max=10):
    inertias, silhouettes, db_scores = [], [], []
    k_range = range(k_min, k_max + 1)

    print(f'Testing K = {k_min} to {k_max} ...')
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init='auto')
        labels = km.fit_predict(X)
        inertias.append(km.inertia_)
        silhouettes.append(silhouette_score(X, labels))
        db_scores.append(davies_bouldin_score(X, labels))
        print(f'  K={k:2d} | Inertia={km.inertia_:,.1f} | Silhouette={silhouettes[-1]:.4f} | DB={db_scores[-1]:.4f}')

    # Auto-select K with highest silhouette
    best_idx = int(np.argmax(silhouettes))
    best_k = list(k_range)[best_idx]

    # --- Plot ---
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Elbow
    axes[0].plot(list(k_range), inertias, marker='o', color='steelblue', linewidth=2)
    axes[0].axvline(best_k, color='red', linestyle='--', alpha=0.7, label=f'Suggested K={best_k}')
    axes[0].set_title('Elbow Method (Inertia)', fontweight='bold')
    axes[0].set_xlabel('Number of Clusters (K)')
    axes[0].set_ylabel('Inertia')
    axes[0].legend()

    # Silhouette
    axes[1].plot(list(k_range), silhouettes, marker='s', color='seagreen', linewidth=2)
    axes[1].axvline(best_k, color='red', linestyle='--', alpha=0.7, label=f'Suggested K={best_k}')
    axes[1].set_title('Silhouette Score (higher = better)', fontweight='bold')
    axes[1].set_xlabel('Number of Clusters (K)')
    axes[1].set_ylabel('Silhouette Score')
    axes[1].legend()

    # Davies-Bouldin
    axes[2].plot(list(k_range), db_scores, marker='^', color='darkorange', linewidth=2)
    axes[2].axvline(best_k, color='red', linestyle='--', alpha=0.7, label=f'Suggested K={best_k}')
    axes[2].set_title('Davies-Bouldin Score (lower = better)', fontweight='bold')
    axes[2].set_xlabel('Number of Clusters (K)')
    axes[2].set_ylabel('Davies-Bouldin Score')
    axes[2].legend()

    plt.suptitle('Cluster Count Evaluation', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('k_selection.png', bbox_inches='tight')
    plt.show()
    print(f'\nSaved: k_selection.png')

    return best_k, inertias, silhouettes


if state['feature_matrix'] is None:
    print('No preprocessed data. Run the Cleaning cell first.')
else:
    best_k, inertias, silhouettes = find_optimal_k(
        state['feature_matrix'], K_MIN, K_MAX
    )
    state['optimal_k'] = best_k
    print(f'\nSuggested K = {best_k} (highest silhouette score)')
    print('You can override by setting state["optimal_k"] = <your choice> before running the next cell.')

---
## 6. K-Means Clustering

Run K-Means with the chosen K. Adjust `state['optimal_k']` above to override the suggestion.

In [ ]:
def run_kmeans(X, k):
    print(f'Fitting K-Means with K={k} ...')
    km = KMeans(n_clusters=k, random_state=42, n_init='auto', max_iter=500)
    labels = km.fit_predict(X)

    sil = silhouette_score(X, labels)
    db  = davies_bouldin_score(X, labels)

    print(f'  Inertia        : {km.inertia_:,.2f}')
    print(f'  Silhouette     : {sil:.4f}  (range -1 to 1; higher is better)')
    print(f'  Davies-Bouldin : {db:.4f}   (lower is better)')

    unique, counts = np.unique(labels, return_counts=True)
    print()
    print('Cluster sizes:')
    for cluster, count in zip(unique, counts):
        pct = count / len(labels) * 100
        print(f'  Cluster {cluster}: {count:,} customers ({pct:.1f}%)')

    return labels, km


if state['feature_matrix'] is None or state['optimal_k'] is None:
    print('Run the Cleaning and K-Selection cells first.')
else:
    labels, km_model = run_kmeans(state['feature_matrix'], state['optimal_k'])
    state['cluster_labels'] = labels
    state['km_model'] = km_model
    # Attach cluster labels to the cleaned dataframe
    state['clean_df']['Cluster'] = labels
    print()
    print('Clustering complete. Run the Visualizations cell next.')

---
## 7. Visualizations

Five plots are generated:
1. **2-D PCA scatter** — clusters in reduced 2-D space
2. **Cluster size distribution** — bar chart
3. **Feature importance per cluster** — heatmap of mean values
4. **Top feature box plots** — distribution per cluster for the most important features
5. **Silhouette plot** — per-sample silhouette coefficients

In [ ]:
def plot_pca_scatter(X, labels, k):
    pca = PCA(n_components=2, random_state=42)
    X_2d = pca.fit_transform(X)
    var = pca.explained_variance_ratio_

    fig, ax = plt.subplots(figsize=(9, 6))
    colors = plt.cm.get_cmap(PALETTE, k)
    for c in range(k):
        mask = labels == c
        ax.scatter(
            X_2d[mask, 0], X_2d[mask, 1],
            s=40, alpha=0.6, color=colors(c), label=f'Cluster {c}',
        )
    ax.set_xlabel(f'PC1 ({var[0]*100:.1f}% variance)', fontsize=11)
    ax.set_ylabel(f'PC2 ({var[1]*100:.1f}% variance)', fontsize=11)
    ax.set_title('Customer Segments — PCA 2-D Projection', fontsize=13, fontweight='bold')
    ax.legend(title='Cluster', loc='best')
    plt.tight_layout()
    plt.savefig('pca_scatter.png', bbox_inches='tight')
    plt.show()
    print('Saved: pca_scatter.png')


def plot_cluster_sizes(labels, k):
    unique, counts = np.unique(labels, return_counts=True)
    colors = [plt.cm.get_cmap(PALETTE, k)(i) for i in range(k)]

    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar([f'Cluster {c}' for c in unique], counts, color=colors, edgecolor='white')
    for bar, cnt in zip(bars, counts):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(counts) * 0.01,
            f'{cnt:,}\n({cnt/len(labels)*100:.1f}%)',
            ha='center', va='bottom', fontsize=10,
        )
    ax.set_title('Cluster Size Distribution', fontsize=13, fontweight='bold')
    ax.set_ylabel('Number of Customers')
    ax.set_ylim(0, max(counts) * 1.18)
    plt.tight_layout()
    plt.savefig('cluster_sizes.png', bbox_inches='tight')
    plt.show()
    print('Saved: cluster_sizes.png')


def plot_feature_heatmap(clean_df, feature_cols, labels, k, top_n=15):
    df_tmp = clean_df[feature_cols].copy()
    df_tmp['Cluster'] = labels
    cluster_means = df_tmp.groupby('Cluster')[feature_cols].mean()

    # Pick the features with highest variance across cluster means
    variances = cluster_means.var(axis=0).sort_values(ascending=False)
    top_features = variances.head(top_n).index.tolist()

    # Z-score normalise means for readability
    heatmap_data = cluster_means[top_features]
    heatmap_z = (heatmap_data - heatmap_data.mean()) / (heatmap_data.std() + 1e-9)

    fig, ax = plt.subplots(figsize=(max(8, len(top_features) * 0.7), max(4, k * 0.8)))
    sns.heatmap(
        heatmap_z, annot=True, fmt='.2f', cmap='RdYlGn',
        linewidths=0.5, ax=ax, cbar_kws={'label': 'Z-score vs cluster mean'},
    )
    ax.set_title(f'Top {top_n} Differentiating Features per Cluster (Z-scored means)',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature')
    ax.set_ylabel('Cluster')
    plt.tight_layout()
    plt.savefig('feature_heatmap.png', bbox_inches='tight')
    plt.show()
    print('Saved: feature_heatmap.png')
    return top_features


def plot_feature_boxplots(clean_df, feature_cols, labels, top_features, k, n_plots=6):
    selected = top_features[:n_plots]
    ncols = 3
    nrows = (len(selected) + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4))
    axes = np.array(axes).flatten()

    colors = [plt.cm.get_cmap(PALETTE, k)(i) for i in range(k)]
    df_tmp = clean_df[feature_cols].copy()
    df_tmp['Cluster'] = labels

    for ax, feat in zip(axes, selected):
        data_by_cluster = [df_tmp[df_tmp['Cluster'] == c][feat].values for c in range(k)]
        bp = ax.boxplot(data_by_cluster, patch_artist=True, notch=False)
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        ax.set_xticklabels([f'C{c}' for c in range(k)])
        ax.set_title(feat, fontsize=10, fontweight='bold')
        ax.set_xlabel('Cluster')

    # Hide unused subplot axes
    for ax in axes[len(selected):]:
        ax.set_visible(False)

    fig.suptitle('Feature Distributions per Cluster (Top Differentiators)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('feature_boxplots.png', bbox_inches='tight')
    plt.show()
    print('Saved: feature_boxplots.png')


def plot_silhouette(X, labels, k):
    from sklearn.metrics import silhouette_samples
    sample_scores = silhouette_samples(X, labels)
    colors = plt.cm.get_cmap(PALETTE, k)

    fig, ax = plt.subplots(figsize=(9, 5))
    y_lower = 10
    for c in range(k):
        cluster_scores = np.sort(sample_scores[labels == c])
        size = len(cluster_scores)
        y_upper = y_lower + size
        ax.fill_betweenx(
            np.arange(y_lower, y_upper), 0, cluster_scores,
            facecolor=colors(c), alpha=0.7, label=f'Cluster {c}',
        )
        ax.text(-0.05, y_lower + size / 2, str(c), fontsize=9)
        y_lower = y_upper + 10

    avg = silhouette_score(X, labels)
    ax.axvline(avg, color='red', linestyle='--', label=f'Avg score = {avg:.3f}')
    ax.set_title('Silhouette Plot — Per-Sample Scores by Cluster',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Silhouette Coefficient')
    ax.set_ylabel('Cluster')
    ax.set_yticks([])
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig('silhouette_plot.png', bbox_inches='tight')
    plt.show()
    print('Saved: silhouette_plot.png')


# Run all visualizations
if state['cluster_labels'] is None:
    print('No clusters yet. Run the Clustering cell first.')
else:
    X      = state['feature_matrix']
    labels = state['cluster_labels']
    k      = state['optimal_k']
    cdf    = state['clean_df']
    fcols  = state['feature_columns']

    print('Generating visualizations...\n')

    plot_pca_scatter(X, labels, k)
    plot_cluster_sizes(labels, k)
    top_feats = plot_feature_heatmap(cdf, fcols, labels, k)
    plot_feature_boxplots(cdf, fcols, labels, top_feats, k)
    plot_silhouette(X, labels, k)

    print('\nAll visualizations saved to the working directory.')

---
## 8. Export Segmented Data

Save the original data with cluster labels appended.

In [ ]:
def export_segmented(output_path='segmented_customers.csv'):
    if state['raw_df'] is None or state['cluster_labels'] is None:
        print('Nothing to export yet.')
        return

    # Align labels to the cleaned (non-duplicate) rows, then merge back
    clean_idx = state['clean_df'].index
    export_df = state['raw_df'].loc[clean_idx].copy()
    export_df['Cluster'] = state['cluster_labels']
    export_df.to_csv(output_path, index=False)
    print(f'Saved {len(export_df):,} rows to "{output_path}"')
    display(export_df.groupby('Cluster').size().rename('count').reset_index())


export_segmented()